# Workshop 03 — Evaluate the BASE Qwen3-4B (before fine-tuning)

**This notebook needs NO fine-tuned model.** Run it any time — even before notebook 01 finishes training — to establish a baseline you can compare against later.

## What you'll do

1. Pull the **validation split** of `gt2026workshop/phreshphish-2k` and register it as a SageMaker dataset.
2. Register a tiny **custom scorer** (Lambda) that gives 1.0 when the model's first word matches the ground-truth label.
3. Run a **Custom Scorer evaluation** against the **base** `huggingface-reasoning-qwen3-4b` — *how good is Qwen3-4B at phishing detection out of the box?*
4. (Optional) Run **MMLU** and **LLM-as-a-Judge** on the base model so you have those baselines too.

Notebook **04** evaluates the fine-tuned model with the **same dataset + scorer**, then compares. We deliberately register the dataset and reward function here once, so notebook 04 just reuses them by name.

## Prerequisites

- The execution role has `AmazonSageMakerModelCustomizationCoreAccess` (or equivalent)
- The SageMaker execution role is **assumable by Lambda** (trust policy includes `lambda.amazonaws.com`) — used for the reward function. Or set a separate `LAMBDA_ROLE_ARN` in config.
- An MLflow app ARN to reuse (so eval metrics are tracked) — see config
- For LLM-as-a-Judge: Bedrock access with the chosen judge model enabled

## 0. Install / upgrade SDK

In [ ]:
%pip install -q --upgrade sagemaker datasets boto3 rich
# Restart the kernel after this cell.

## 1. Configuration

These are shared with notebook 04 — keep them identical across both so the comparison is apples-to-apples.

In [ ]:
HF_DATASET_ID = "gt2026workshop/phreshphish-2k"
HF_CONFIG_NAME = "text"
# S3 output — leave S3_BUCKET = None to use the account's DEFAULT SageMaker bucket
# (sagemaker-<region>-<account-id>, auto-created on first use). The session cell
# resolves S3_OUTPUT_PATH = s3://<bucket>/<prefix>/ once the session exists.
S3_BUCKET = None
S3_PREFIX = "qwen3-4b-phish-sft"
ROLE_ARN = None  # SageMaker execution role; auto-detected if None

BASE_MODEL = "huggingface-reasoning-qwen3-4b"   # the untouched base model

# Shared names — notebook 04 reuses these exact names via .get()
EVAL_DATASET_NAME = "phreshphish-eval-100"
REWARD_EVALUATOR_NAME = "phish-label-exact-match"

EVAL_LIMIT = 100   # validation rows to score (full val is 200)

# Role the reward-function Lambda runs as. The SageMaker execution role works
# IF its trust policy allows lambda.amazonaws.com (we've made ours assumable).
# Leave None to reuse ROLE_ARN; set an explicit ARN only if you keep a separate role.
LAMBDA_ROLE_ARN = None

# Reuse an EXISTING MLflow app (auto-discovered below if None). Avoids the
# 3-app quota error you hit when the SDK tries to auto-create one.
MLFLOW_RESOURCE_ARN = None

# Bedrock judge model — must be enabled in your account/region
JUDGE_MODEL = "anthropic.claude-3-5-haiku-20241022-v1:0"

## 2. SageMaker session + MLflow discovery

In [ ]:
import os, boto3
from sagemaker.core.helper.session_helper import Session
from rich import print as rprint
from rich.pretty import pprint

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)
sagemaker_session = Session(sagemaker_client=sm_client)

# Default SageMaker bucket (sagemaker-<region>-<acct>); override via S3_BUCKET above.
BUCKET = S3_BUCKET or sagemaker_session.default_bucket()
S3_OUTPUT_PATH = f"s3://{BUCKET}/{S3_PREFIX}/"


if ROLE_ARN is None:
    from sagemaker.core.helper.session_helper import get_execution_role
    ROLE_ARN = get_execution_role()


def discover_mlflow_app(session_client):
    try:
        apps = []
        for page in session_client.get_paginator("list_mlflow_apps").paginate():
            apps.extend(page.get("Summaries", []))
    except Exception as e:
        print(f"Could not list MLflow apps ({e}).")
        return None
    ready = [a for a in apps if a.get("Status") in ("Created", "Updated")]
    for a in apps:
        print(f"  - {a.get('Name')}  status={a.get('Status')}  arn={a.get('Arn')}")
    return ready[0]["Arn"] if ready else None


if MLFLOW_RESOURCE_ARN is None:
    MLFLOW_RESOURCE_ARN = discover_mlflow_app(sm_client)

rprint({"region": REGION, "role": ROLE_ARN, "output": S3_OUTPUT_PATH, "mlflow": MLFLOW_RESOURCE_ARN})

## 3. Stage + register the validation dataset

Each row has `prompt` (the classification query) and `completion` (`phish`/`benign` ground truth) — the HuggingFace Prompt-Completion format SageMaker eval accepts. We register it once here; notebook 04 reuses it by name.

In [ ]:
import json, pathlib
from datasets import load_dataset
from urllib.parse import urlparse

ds = load_dataset(HF_DATASET_ID, name=HF_CONFIG_NAME, split="validation")
if EVAL_LIMIT and EVAL_LIMIT < len(ds):
    ds = ds.select(range(EVAL_LIMIT))
print(f"eval rows: {len(ds)}")

p = urlparse(S3_OUTPUT_PATH.rstrip("/"))
BUCKET = p.netloc
PREFIX = p.path.lstrip("/")
EVAL_PREFIX = f"{PREFIX}/eval"

tmp = pathlib.Path("./_eval_staged"); tmp.mkdir(exist_ok=True)
eval_jsonl = tmp / "eval.jsonl"
with open(eval_jsonl, "w") as f:
    for r in ds:
        f.write(json.dumps({"prompt": r["prompt"], "completion": r["completion"]}) + "\n")
print(f"wrote {eval_jsonl}  ({eval_jsonl.stat().st_size:,} bytes)")

s3 = boto3.client("s3")
EVAL_S3 = f"s3://{BUCKET}/{EVAL_PREFIX}/eval.jsonl"
s3.upload_file(str(eval_jsonl), BUCKET, f"{EVAL_PREFIX}/eval.jsonl")
print(f"S3: {EVAL_S3}")

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

# get-or-create: reuse if a dataset with this name already exists
try:
    eval_dataset = DataSet.get(name=EVAL_DATASET_NAME, sagemaker_session=sagemaker_session)
    print("reusing existing dataset")
except Exception:
    eval_dataset = DataSet.create(name=EVAL_DATASET_NAME, source=EVAL_S3, wait=True)
    print("created dataset")
rprint({"eval_dataset_arn": eval_dataset.arn})

## 4. Register the custom scorer (reward function)

Returns 1.0 when the model's first word matches the truth label (`phish`/`benign`), else 0.0. The handler tolerates the several payload shapes SageMaker's scorer may send.

> SageMaker creates a Lambda for the scorer, and that Lambda needs a role whose **trust policy allows `lambda.amazonaws.com`**. We've made the SageMaker execution role assumable by Lambda, so `LAMBDA_ROLE_ARN = None` reuses `ROLE_ARN`. If your execution role is *not* Lambda-assumable you'll see *"The role defined for the function cannot be assumed by Lambda"* — set `LAMBDA_ROLE_ARN` to a dedicated role instead (see README).

> **Why the reward function strips `<think>`:** `huggingface-reasoning-qwen3-4b` is a *reasoning* model. Its raw eval output is a **stringified Python list** wrapping a chain-of-thought, e.g. `"['<think>\n...\n</think>\n\nphish']"`. A naive "first word" scorer reads `think` and scores **0% for every sample** — which looks like a broken model but is really a broken scorer. `extract_label()` below unwraps the list, removes the `<think>...</think>` block, then takes the first `phish`/`benign` token. Verified against real job output: base ≈ 81%, fine-tuned ≈ 98%.

In [ ]:
REWARD_FUNCTION_SOURCE = "\nimport json\nimport re\nimport uuid\nimport ast\nfrom typing import Any, Dict, List, Optional\n\nVALID_LABELS = {\"phish\", \"benign\"}\n_THINK_CLOSED = re.compile(r\"<think>.*?</think>\", re.DOTALL | re.IGNORECASE)\n_THINK_OPEN = re.compile(r\"<think>.*\", re.DOTALL | re.IGNORECASE)\n_LABEL = re.compile(r\"(phish|benign)\", re.IGNORECASE)\n\n\ndef extract_label(text: str) -> str:\n    \"\"\"Pull a clean phish/benign label out of a (possibly messy) model output.\n\n    Handles two real quirks of huggingface-reasoning-qwen3-4b eval output:\n      1) the inference arrives as a stringified Python list, e.g. \"['...']\"\n      2) the model wraps a (often empty) chain-of-thought in <think>...</think>\n    The label after </think> is the model's actual answer, so we strip the\n    think block first, then take the FIRST label token that remains.\n    \"\"\"\n    if not text:\n        return \"\"\n    try:\n        v = ast.literal_eval(text)\n        if isinstance(v, list) and v:\n            text = str(v[0])\n    except Exception:\n        pass\n    text = _THINK_CLOSED.sub(\" \", text)\n    text = _THINK_OPEN.sub(\" \", text)\n    m = _LABEL.search(text)\n    return m.group(1).lower() if m else \"\"\n\n\ndef custom_reward(prediction: str, ground_truth: str) -> float:\n    pred = extract_label(prediction)\n    truth = extract_label(ground_truth) or (ground_truth or \"\").strip().lower()\n    if pred not in VALID_LABELS:\n        return 0.0\n    return 1.0 if pred == truth else 0.0\n\n\ndef _ok(body, code=200):\n    return {\"statusCode\": code, \"headers\": {\"content-type\": \"application/json\"}, \"body\": json.dumps(body)}\n\n\ndef _prediction_text(sample):\n    msgs = sample.get(\"messages\")\n    if isinstance(msgs, list):\n        for m in reversed(msgs):\n            if isinstance(m, dict) and m.get(\"role\") == \"assistant\":\n                c = m.get(\"content\")\n                if isinstance(c, str):\n                    return c\n                if isinstance(c, list):\n                    for part in c:\n                        if isinstance(part, dict) and part.get(\"type\") == \"text\":\n                            return part.get(\"text\") or \"\"\n    for k in (\"inference\", \"prediction\", \"response\", \"model_output\", \"completion\", \"generated_text\", \"output\"):\n        v = sample.get(k)\n        if isinstance(v, str) and v.strip():\n            return v\n        if isinstance(v, list) and v:\n            return str(v[0])\n    return \"\"\n\n\ndef _ground_truth(sample):\n    for k in (\"reference_answer\", \"ground_truth\", \"completion\", \"answer\", \"label\"):\n        v = sample.get(k)\n        if isinstance(v, str) and v.strip():\n            return v.strip()\n    md = sample.get(\"metadata\") or {}\n    if isinstance(md, dict):\n        for k in (\"reference_answer\", \"ground_truth\", \"answer\"):\n            v = md.get(k)\n            if isinstance(v, str) and v.strip():\n                return v.strip()\n    return \"\"\n\n\ndef _score_one(sample):\n    sid = sample.get(\"id\") or str(uuid.uuid4())\n    pred = _prediction_text(sample)\n    gt = _ground_truth(sample)\n    score = custom_reward(pred, gt) if (pred and gt) else 0.0\n    return {\n        \"id\": sid,\n        \"aggregate_reward_score\": float(score),\n        \"metrics_list\": [{\"name\": \"label_exact_match\", \"value\": float(score), \"type\": \"Reward\"}],\n    }\n\n\ndef _extract_samples(event):\n    if isinstance(event, list):\n        return event\n    if not isinstance(event, dict):\n        return None\n    if event.get(\"requestContext\", {}).get(\"http\", {}).get(\"method\") == \"OPTIONS\":\n        return None\n    body = event.get(\"body\")\n    if isinstance(body, str):\n        try:\n            body = json.loads(body)\n        except json.JSONDecodeError:\n            return None\n    if isinstance(body, list):\n        return body\n    if isinstance(body, dict) and isinstance(body.get(\"batch\"), list):\n        return body[\"batch\"]\n    if isinstance(event.get(\"batch\"), list):\n        return event[\"batch\"]\n    return None\n\n\ndef lambda_handler(event, context):\n    if isinstance(event, dict) and event.get(\"requestContext\", {}).get(\"http\", {}).get(\"method\") == \"OPTIONS\":\n        return _ok({\"ok\": True})\n    samples = _extract_samples(event)\n    if samples is None:\n        return _ok({\"error\": \"could not parse event of type \" + type(event).__name__}, 400)\n    try:\n        results = [_score_one(s) for s in samples]\n    except Exception as e:\n        return _ok({\"error\": \"scoring failed: \" + str(e)}, 500)\n    return _ok(results)\n"

REWARD_PATH = pathlib.Path("./reward_phishing_label.py")
REWARD_PATH.write_text(REWARD_FUNCTION_SOURCE)
print(f"wrote {REWARD_PATH}")

In [ ]:
from sagemaker.ai_registry.evaluator import Evaluator
from sagemaker.ai_registry.air_constants import REWARD_FUNCTION

# get-or-create the evaluator so re-runs (and notebook 04) reuse it
try:
    phish_evaluator = Evaluator.get(name=REWARD_EVALUATOR_NAME, sagemaker_session=sagemaker_session)
    print("reusing existing evaluator")
except Exception:
    phish_evaluator = Evaluator.create(
        name=REWARD_EVALUATOR_NAME,
        source=str(REWARD_PATH),
        type=REWARD_FUNCTION,
        role=LAMBDA_ROLE_ARN or ROLE_ARN,   # exec role works if it's Lambda-assumable
    )
    print("created evaluator")
rprint({"evaluator_arn": phish_evaluator.arn})

## 5. Custom Scorer evaluation — BASE model only

We pass `model=BASE_MODEL` (the JumpStart id string) and `evaluate_base_model=False`. There is no fine-tuned model package involved, so this runs standalone.

Typical runtime: 5–10 minutes (mostly inference-worker startup).

In [ ]:
from sagemaker.train.evaluate import CustomScorerEvaluator

base_scorer = CustomScorerEvaluator(
    evaluator=phish_evaluator,
    model=BASE_MODEL,                  # base JumpStart model — no fine-tuning needed
    dataset=eval_dataset.arn,
    s3_output_path=f"{S3_OUTPUT_PATH}eval/base/custom-scorer/",
    evaluate_base_model=False,
    mlflow_resource_arn=MLFLOW_RESOURCE_ARN,
    mlflow_experiment_name="qwen3-4b-phish-eval",
    mlflow_run_name="base-custom-scorer",
)

base_execution = base_scorer.evaluate()
print("job:", base_execution.name)

In [ ]:
base_execution.wait(target_status="Succeeded", poll=15, timeout=3600)
print("final status:", base_execution.status.overall_status)
base_execution.show_results()

## 5b. Score from artifacts (the trustworthy accuracy)

⚠️ **The `label_exact_match` printed above is not reliable for this reasoning model.** `huggingface-reasoning-qwen3-4b` emits a `<think>…</think>` block, and the managed Custom-Scorer pipeline mis-records the per-sample reward (it reports 100% even when the model is wrong — verified against the raw outputs). 

Instead we compute accuracy **directly from the inference artifacts** the job wrote to S3 (`inference_output.jsonl`), joining each model output to its ground-truth label and parsing the real `phish`/`benign` answer out of the reasoning trace. This is the number to trust and to quote in the workshop.

In [ ]:
import re, ast, json, hashlib, boto3
from urllib.parse import urlparse
from collections import Counter

_THINK_CLOSED = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_THINK_OPEN   = re.compile(r"<think>.*", re.DOTALL | re.IGNORECASE)
_LABEL        = re.compile(r"(phish|benign)", re.IGNORECASE)
_URL          = re.compile(r"URL:\s*((?:https?://)?[^\s\\\"]+)")
_WS           = re.compile(r"\s+")

def extract_label(text):
    """Pull the clean phish/benign answer out of a reasoning-model output.
    Handles: stringified-list wrapper "['...']", and the <think>...</think> block
    (the answer is AFTER </think>, so strip the think block first)."""
    if not text:
        return ""
    try:
        v = ast.literal_eval(text)
        if isinstance(v, list) and v:
            text = str(v[0])
    except Exception:
        pass
    text = _THINK_CLOSED.sub(" ", text)
    text = _THINK_OPEN.sub(" ", text)
    m = _LABEL.search(text)
    return m.group(1).lower() if m else ""

def join_key(prompt):
    """Stable key to join a prediction to its gold row (eval shuffles order)."""
    if not prompt:
        return None
    m = _URL.search(prompt)
    if m:
        return "url:" + m.group(1)
    body = prompt.split("---", 1)[-1] if "---" in prompt else prompt
    body = _WS.sub(" ", body).strip().lower()[:600]
    return "h:" + hashlib.sha1(body.encode("utf-8")).hexdigest()[:16]

def _split(uri):
    u = urlparse(uri); return u.netloc, u.path.lstrip("/")

def _read_jsonl(s3, bucket, key):
    data = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8")
    return [json.loads(l) for l in data.splitlines() if l.strip()]

def find_inference_output(s3, output_prefix_uri):
    """Find inference_output.jsonl under an eval run's S3 output prefix."""
    bucket, prefix = _split(output_prefix_uri)
    hits = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].rsplit("/", 1)[-1] == "inference_output.jsonl":
                hits.append(obj["Key"])
    if not hits:
        raise FileNotFoundError(f"no inference_output.jsonl under {output_prefix_uri} yet")
    return f"s3://{bucket}/{sorted(hits)[-1]}"

def score_from_artifacts(inference_s3_uri, ground_truth_s3_uri):
    s3 = boto3.client("s3")
    ib, ik = _split(inference_s3_uri); gb, gk = _split(ground_truth_s3_uri)
    truth = {}
    for r in _read_jsonl(s3, gb, gk):
        k = join_key(r.get("prompt", ""))
        if k:
            truth[k] = extract_label(r.get("completion", "")) or (r.get("completion") or "").strip().lower()
    matched = correct = no_label = unmatched = 0
    conf = Counter()
    for r in _read_jsonl(s3, ib, ik):
        k = join_key(r.get("prompt", "")); pred = extract_label(r.get("inference", ""))
        if k not in truth:
            unmatched += 1; continue
        matched += 1
        if not pred: no_label += 1
        conf[(truth[k], pred or "??")] += 1
        if pred == truth[k]: correct += 1
    # confusion keyed by "truth>pred" so it is JSON-serializable
    confusion = {f"{t}>{p}": n for (t, p), n in conf.items()}
    return {"matched": matched, "unmatched": unmatched, "correct": correct,
            "no_label": no_label, "accuracy": correct / matched if matched else 0.0,
            "confusion": confusion}

def print_report(name, res):
    c = res["confusion"]; g = lambda t, p: c.get(f"{t}>{p}", 0)
    tp, tn, fp, fn = g("phish","phish"), g("benign","benign"), g("benign","phish"), g("phish","benign")
    print(f"\n{'='*46}\n{name}\n{'='*46}")
    print(f"  accuracy:  {res['accuracy']:.1%}  ({res['correct']}/{res['matched']})")
    if res["unmatched"]: print(f"  unmatched: {res['unmatched']}")
    if res["no_label"]:  print(f"  no_label:  {res['no_label']}")
    print("\n  confusion (rows=truth, cols=pred):")
    print("                pred=phish   pred=benign")
    print(f"    phish          {tp:>4}         {fn:>4}")
    print(f"    benign         {fp:>4}         {tn:>4}")
    pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    print(f"\n  phish  precision={pr:.1%}  recall={rc:.1%}  f1={f1:.1%}")

# Ground truth = the JSONL we staged in section 3 (prompt + completion)
GROUND_TRUTH_S3 = EVAL_S3

# Locate this run's inference output under the prefix we gave the evaluator
BASE_OUTPUT_PREFIX = f"{S3_OUTPUT_PATH}eval/base/custom-scorer/"
s3 = boto3.client("s3")
base_inf = find_inference_output(s3, BASE_OUTPUT_PREFIX)
print("scoring:", base_inf)

base_metrics = score_from_artifacts(base_inf, GROUND_TRUTH_S3)
print_report("BASE Qwen3-4B — true accuracy", base_metrics)

# Stash for notebook 04 to compare against (also save to S3 for cross-notebook reuse)
import json as _json
s3.put_object(Bucket=_split(S3_OUTPUT_PATH)[0],
              Key=_split(S3_OUTPUT_PATH)[1] + "eval/base/true_metrics.json",
              Body=_json.dumps(base_metrics).encode())


The table from `show_results()` includes `label_exact_match`, but **trust the artifact-based number from section 5b instead** — see the warning there. Base Qwen3-4B lands around **80%**; it tends to *under*-detect phishing (false negatives), which notebook 04 shows the fine-tune fixes.

## 6. (Optional) MMLU on the base model

Baseline general-knowledge score, so notebook 04's MMLU run can show whether fine-tuning eroded it. A single subtask keeps it to ~5 minutes.

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks

Benchmark = get_benchmarks()

base_mmlu = BenchMarkEvaluator(
    benchmark=Benchmark.MMLU,
    model=BASE_MODEL,
    subtasks=["computer_security"],   # phishing-relevant, ~100 questions
    s3_output_path=f"{S3_OUTPUT_PATH}eval/base/mmlu/",
    evaluate_base_model=False,
    mlflow_resource_arn=MLFLOW_RESOURCE_ARN,
    mlflow_experiment_name="qwen3-4b-phish-eval",
    mlflow_run_name="base-mmlu",
)
base_mmlu_exec = base_mmlu.evaluate()
print("job:", base_mmlu_exec.name)

In [ ]:
base_mmlu_exec.wait(target_status="Succeeded", poll=30, timeout=7200)
print("final status:", base_mmlu_exec.status.overall_status)
base_mmlu_exec.show_results()

## 7. (Optional) LLM-as-a-Judge on the base model

Requires Bedrock access + the chosen judge model enabled in your region. Skip otherwise.

In [ ]:
import json
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

custom_metric = {
    "customMetricDefinition": {
        "name": "PhishingLabelAgreement",
        "instructions": (
            "You are evaluating a phishing-detection classifier. Given the input page summary in "
            "`prompt` and the model's prediction in `prediction`, decide whether the prediction's "
            "first meaningful word matches the correct label (phish or benign).\n\n"
            "Prompt: {{prompt}}\nPrediction: {{prediction}}"
        ),
        "ratingScale": [
            {"definition": "Agree", "value": {"floatValue": 1}},
            {"definition": "Disagree", "value": {"floatValue": 0}},
        ],
    }
}

base_llmaj = LLMAsJudgeEvaluator(
    model=BASE_MODEL,
    evaluator_model=JUDGE_MODEL,
    dataset=EVAL_S3,
    builtin_metrics=["Faithfulness"],
    custom_metrics=json.dumps([custom_metric]),
    s3_output_path=f"{S3_OUTPUT_PATH}eval/base/llmaj/",
    evaluate_base_model=False,
    mlflow_resource_arn=MLFLOW_RESOURCE_ARN,
    mlflow_experiment_name="qwen3-4b-phish-eval",
    mlflow_run_name="base-llmaj",
)
base_llmaj_exec = base_llmaj.evaluate()
print("job:", base_llmaj_exec.name)

In [ ]:
base_llmaj_exec.wait(target_status="Succeeded", poll=30, timeout=7200)
print("final status:", base_llmaj_exec.status.overall_status)
base_llmaj_exec.show_results()

## 8. Next

Once notebook 01 has produced a fine-tuned model package, run **`04_evaluate_finetuned_model.ipynb`**. It reuses the dataset and reward function registered here (same names), scores the fine-tuned model, and shows the lift over this baseline.

All runs are visible together in **MLflow** under experiment `qwen3-4b-phish-eval`, and in the **Studio model card → Evaluations** tab.